# 🌸 Clasificación de Flores Iris
## El "Hola Mundo" del Machine Learning

---

En este ejercicio vamos a entrenar un modelo de **Machine Learning** que aprende a identificar tres especies de flores **Iris** según sus medidas físicas.

### 🌺 Las tres especies:
| Especie | Descripción |
|---|---|
| *Iris setosa* | Pétalos pequeños y compactos |
| *Iris versicolor* | Tamaño intermedio |
| *Iris virginica* | Pétalos grandes y alargados |

### 📏 Las cuatro características (features):
- Largo del sépalo (cm)
- Ancho del sépalo (cm)
- Largo del pétalo (cm)
- Ancho del pétalo (cm)

### 🎯 Objetivos:
- Cargar y explorar un dataset real
- Visualizar los datos
- Entrenar distintos modelos de clasificación
- Comparar su rendimiento
- Hacer predicciones con datos nuevos

### 🗺️ Estructura:
1. Importar librerías
2. Cargar y explorar el dataset
3. Visualizar los datos
4. Preparar los datos
5. Entrenar modelos
6. Evaluar y comparar
7. Hacer predicciones
8. 🏆 Desafíos extra

---
## 📦 Paso 1: Importar librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset
from sklearn.datasets import load_iris

# Preprocesamiento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Modelos
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Evaluación
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print('✅ Librerías importadas correctamente')

---
## 📂 Paso 2: Cargar y explorar el dataset

El dataset Iris viene incluido en scikit-learn. Tiene **150 muestras** (50 por especie).

In [ ]:
# Cargar el dataset
iris = load_iris()

# Convertir a DataFrame para explorar más fácil
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['especie'] = pd.Categorical.from_codes(iris.target, iris.target_names)
df['especie_id'] = iris.target

print('📊 Primeras 10 filas del dataset:')
print(df.head(10))
print(f'\nDimensiones: {df.shape}  →  {df.shape[0]} flores, {df.shape[1]-2} características')
print(f'Especies: {list(iris.target_names)}')

In [ ]:
# Estadísticas descriptivas
print('📈 Estadísticas descriptivas por característica:')
df.drop(columns=['especie_id']).groupby('especie').mean().round(2)

In [ ]:
# Distribución de clases
print('🌸 Cantidad de flores por especie:')
print(df['especie'].value_counts())
print('\n✅ Dataset perfectamente balanceado: 50 flores de cada especie')

---
## 📊 Paso 3: Visualizar los datos

Antes de entrenar cualquier modelo, es clave **entender los datos visualmente**.

In [ ]:
# Colores para cada especie
colores = {'setosa': '#E74C3C', 'versicolor': '#2ECC71', 'virginica': '#3498DB'}

# Gráfico de dispersión: pétalo largo vs pétalo ancho
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Exploración del dataset Iris', fontsize=14, fontweight='bold')

# Largo vs Ancho de pétalo
for especie, color in colores.items():
    subset = df[df['especie'] == especie]
    axes[0].scatter(
        subset['petal length (cm)'], subset['petal width (cm)'],
        label=especie, color=color, alpha=0.7, edgecolors='white', s=80
    )
axes[0].set_xlabel('Largo del pétalo (cm)')
axes[0].set_ylabel('Ancho del pétalo (cm)')
axes[0].set_title('Pétalos: Largo vs Ancho')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Largo vs Ancho de sépalo
for especie, color in colores.items():
    subset = df[df['especie'] == especie]
    axes[1].scatter(
        subset['sepal length (cm)'], subset['sepal width (cm)'],
        label=especie, color=color, alpha=0.7, edgecolors='white', s=80
    )
axes[1].set_xlabel('Largo del sépalo (cm)')
axes[1].set_ylabel('Ancho del sépalo (cm)')
axes[1].set_title('Sépalos: Largo vs Ancho')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('💡 ¿Notás que las flores se agrupan de forma natural?')
print('   ¡Eso es lo que el modelo va a aprender a detectar!')

In [ ]:
# Boxplots: distribución de cada característica por especie
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Distribución de características por especie', fontsize=14, fontweight='bold')

caracteristicas = [
    ('sepal length (cm)', 'Largo del sépalo'),
    ('sepal width (cm)',  'Ancho del sépalo'),
    ('petal length (cm)', 'Largo del pétalo'),
    ('petal width (cm)',  'Ancho del pétalo'),
]

for ax, (col, titulo) in zip(axes.flat, caracteristicas):
    sns.boxplot(
        data=df, x='especie', y=col, ax=ax,
        palette=colores
    )
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('cm')
    ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Mapa de correlación entre características
fig, ax = plt.subplots(figsize=(7, 5))
corr = df[iris.feature_names].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            xticklabels=['Sep.Largo','Sep.Ancho','Pet.Largo','Pet.Ancho'],
            yticklabels=['Sep.Largo','Sep.Ancho','Pet.Largo','Pet.Ancho'],
            ax=ax)
ax.set_title('Correlación entre características', fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Valores cercanos a 1 o -1 indican alta correlación entre características.')

---
## ⚙️ Paso 4: Preparar los datos

Separamos el dataset en **entrenamiento** (80%) y **prueba** (20%).

In [ ]:
# Separar features (X) y etiquetas (y)
X = iris.data
y = iris.target

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% para prueba
    random_state=42,
    stratify=y          # Mantener proporción de clases
)

# Escalar los datos (importante para algunos modelos como KNN y SVM)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('📊 División del dataset:')
print(f'   Entrenamiento: {X_train.shape[0]} flores')
print(f'   Prueba:        {X_test.shape[0]} flores')
print(f'\n✅ Datos listos para entrenar')

---
## 🤖 Paso 5: Entrenar modelos

Vamos a entrenar **4 modelos distintos** y luego comparar cuál funciona mejor:

| Modelo | Idea principal |
|---|---|
| **KNN** | Clasifica según los vecinos más cercanos |
| **Árbol de Decisión** | Hace preguntas tipo "¿el pétalo mide más de X?" |
| **Random Forest** | Combina muchos árboles de decisión |
| **SVM** | Busca la mejor línea que separa las clases |

In [ ]:
# Definir los modelos
modelos = {
    'KNN (k=5)':            KNeighborsClassifier(n_neighbors=5),
    'Árbol de Decisión':    DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':                  SVC(kernel='rbf', random_state=42)
}

resultados = {}

for nombre, modelo in modelos.items():
    modelo.fit(X_train_sc, y_train)           # Entrenar
    y_pred = modelo.predict(X_test_sc)        # Predecir
    acc = accuracy_score(y_test, y_pred)      # Calcular accuracy
    resultados[nombre] = acc
    print(f'   {nombre:<25} → Accuracy: {acc*100:.1f}%')

print('\n✅ Todos los modelos entrenados')

---
## 📊 Paso 6: Evaluar y comparar modelos

In [ ]:
# Comparación visual de accuracy
fig, ax = plt.subplots(figsize=(9, 5))

nombres = list(resultados.keys())
valores = [v * 100 for v in resultados.values()]
colores_barras = ['#3498DB', '#E74C3C', '#2ECC71', '#9B59B6']

bars = ax.bar(nombres, valores, color=colores_barras, edgecolor='white', width=0.5)
ax.set_title('Comparación de modelos — Accuracy en prueba', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim([80, 105])
ax.grid(True, axis='y', alpha=0.3)

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

mejor = max(resultados, key=resultados.get)
print(f'🏆 Mejor modelo: {mejor} ({resultados[mejor]*100:.1f}%)')

In [ ]:
# Matriz de confusión del mejor modelo
mejor_modelo = modelos[mejor]
y_pred_mejor = mejor_modelo.predict(X_test_sc)

cm = confusion_matrix(y_test, y_pred_mejor)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names, ax=ax)
ax.set_title(f'Matriz de Confusión — {mejor}', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
plt.tight_layout()
plt.show()

print(f'\nReporte completo ({mejor}):\n')
print(classification_report(y_test, y_pred_mejor, target_names=iris.target_names))

In [ ]:
# Visualizar el Árbol de Decisión (muy interpretable)
arbol = modelos['Árbol de Decisión']

fig, ax = plt.subplots(figsize=(16, 7))
plot_tree(
    arbol,
    feature_names=['Sep.Largo', 'Sep.Ancho', 'Pet.Largo', 'Pet.Ancho'],
    class_names=iris.target_names,
    filled=True, rounded=True, fontsize=10, ax=ax
)
ax.set_title('Árbol de Decisión — Reglas aprendidas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Este árbol muestra exactamente las reglas que el modelo aprendió.')
print('   Podés seguir el camino desde arriba hacia abajo para entender cada decisión.')

---
## 🔮 Paso 7: Hacer predicciones con datos nuevos

¡Ahora vamos a usar el modelo para predecir flores que nunca vio!

In [ ]:
def predecir_flor(sep_largo, sep_ancho, pet_largo, pet_ancho, modelo_nombre=None):
    """
    Predice la especie de una flor a partir de sus medidas.

    Parámetros (en cm):
        sep_largo : largo del sépalo
        sep_ancho : ancho del sépalo
        pet_largo : largo del pétalo
        pet_ancho : ancho del pétalo
        modelo_nombre : nombre del modelo a usar (por defecto: el mejor)
    """
    if modelo_nombre is None:
        modelo_nombre = mejor

    m = modelos[modelo_nombre]
    flor = np.array([[sep_largo, sep_ancho, pet_largo, pet_ancho]])
    flor_sc = scaler.transform(flor)

    pred = m.predict(flor_sc)[0]
    especie = iris.target_names[pred]

    # Probabilidades (si el modelo las soporta)
    if hasattr(m, 'predict_proba'):
        probs = m.predict_proba(flor_sc)[0]
        prob_str = ' | '.join([f'{iris.target_names[i]}: {p*100:.1f}%' for i, p in enumerate(probs)])
    else:
        prob_str = 'Modelo sin probabilidades'

    print(f'🌸 Medidas ingresadas:')
    print(f'   Sépalo: {sep_largo} cm largo × {sep_ancho} cm ancho')
    print(f'   Pétalo: {pet_largo} cm largo × {pet_ancho} cm ancho')
    print(f'\n🤖 Modelo: {modelo_nombre}')
    print(f'   Predicción: {especie.upper()}')
    print(f'   Probabilidades: {prob_str}')
    print()


# Probar con flores de ejemplo
print('=== PREDICCIONES DE EJEMPLO ===\n')

# Setosa típica (pétalos pequeños)
predecir_flor(5.0, 3.5, 1.4, 0.2)

# Versicolor típica
predecir_flor(6.0, 2.9, 4.5, 1.5)

# Virginica típica (pétalos grandes)
predecir_flor(6.9, 3.1, 5.8, 2.3)

In [ ]:
# 🔧 ¡Probá con tus propias medidas!
# Valores de referencia:
#   Setosa:     sépalo ~5.0x3.4  pétalo ~1.5x0.3
#   Versicolor: sépalo ~5.9x2.8  pétalo ~4.3x1.3
#   Virginica:  sépalo ~6.6x3.0  pétalo ~5.6x2.0

predecir_flor(
    sep_largo = 5.5,   # ← modificá
    sep_ancho = 2.6,   # ← modificá
    pet_largo = 4.4,   # ← modificá
    pet_ancho = 1.2    # ← modificá
)

---
## 🏆 Desafíos Extra

---
### 🥉 Desafío 1 (Fácil): Cambiar el valor de K en KNN

En KNN, `k` es la cantidad de vecinos que el modelo consulta para decidir. ¿Qué pasa si cambiás ese valor?

In [ ]:
# DESAFÍO 1: Probar distintos valores de K en KNN

k_valores = range(1, 21)
accuracies = []

for k in k_valores:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_sc, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test_sc))
    accuracies.append(acc * 100)

plt.figure(figsize=(9, 4))
plt.plot(k_valores, accuracies, marker='o', color='steelblue', linewidth=2)
plt.title('Accuracy de KNN según el valor de K', fontweight='bold')
plt.xlabel('Valor de K (vecinos)')
plt.ylabel('Accuracy (%)')
plt.xticks(k_valores)
plt.grid(True, alpha=0.3)
plt.ylim([80, 105])
plt.tight_layout()
plt.show()

mejor_k = k_valores[np.argmax(accuracies)]
print(f'🏆 Mejor K: {mejor_k} → Accuracy: {max(accuracies):.1f}%')
print('💡 ¿Notás que con K muy chico o muy grande el accuracy baja?')

---
### 🥈 Desafío 2 (Medio): Importancia de características

Random Forest puede decirnos **qué características son más importantes** para clasificar las flores.

In [ ]:
# DESAFÍO 2: Importancia de características con Random Forest

rf = modelos['Random Forest']
importancias = rf.feature_importances_
nombres_feat = ['Sep. Largo', 'Sep. Ancho', 'Pet. Largo', 'Pet. Ancho']

# Ordenar de mayor a menor
orden = np.argsort(importancias)[::-1]

plt.figure(figsize=(8, 4))
bars = plt.bar(
    [nombres_feat[i] for i in orden],
    [importancias[i] * 100 for i in orden],
    color=['#2ECC71', '#3498DB', '#E74C3C', '#9B59B6'],
    edgecolor='white'
)
for bar, val in zip(bars, sorted(importancias * 100, reverse=True)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', fontweight='bold')
plt.title('Importancia de características (Random Forest)', fontweight='bold')
plt.ylabel('Importancia (%)')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

mas_imp = nombres_feat[orden[0]]
print(f'🌟 Característica más importante: {mas_imp} ({importancias[orden[0]]*100:.1f}%)')
print('💡 ¿Coincide con lo que viste en los gráficos del Paso 3?')

---
### 🥇 Desafío 3 (Difícil): Validación cruzada

La **validación cruzada** (cross-validation) es una técnica más robusta para evaluar modelos, que divide los datos en múltiples particiones y promedia los resultados.

In [ ]:
# DESAFÍO 3: Validación cruzada con k-fold

from sklearn.model_selection import cross_val_score

print('📊 Validación cruzada (5 folds) por modelo:\n')

cv_resultados = {}
for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, scaler.transform(X), y, cv=5, scoring='accuracy')
    cv_resultados[nombre] = scores
    print(f'{nombre:<25} → Media: {scores.mean()*100:.2f}%  ±  {scores.std()*100:.2f}%')

# Boxplot de resultados
fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(
    [v * 100 for v in cv_resultados.values()],
    labels=cv_resultados.keys(),
    patch_artist=True,
    boxprops=dict(facecolor='lightblue')
)
ax.set_title('Distribución de Accuracy — Validación Cruzada (5 folds)', fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\n💡 La validación cruzada da una estimación más confiable')
print('   porque el modelo se evalúa en distintas particiones del dataset.')

---
## 📝 Resumen y Conclusiones

### Lo que aprendiste:

| Concepto | Descripción |
|---|---|
| **Dataset Iris** | 150 flores con 4 características cada una |
| **Train/Test Split** | División para entrenar y evaluar sin sesgo |
| **Normalización** | Escalar datos para que los modelos funcionen mejor |
| **KNN** | Clasifica por similitud con los vecinos más cercanos |
| **Árbol de Decisión** | Aprende reglas interpretables paso a paso |
| **Random Forest** | Ensemble de árboles, más robusto |
| **SVM** | Encuentra el hiperplano óptimo de separación |
| **Accuracy** | Porcentaje de aciertos del modelo |
| **Matriz de confusión** | Detalle de aciertos y errores por clase |
| **Validación cruzada** | Evaluación más confiable con múltiples particiones |

### Resultados típicos esperados:
- Todos los modelos superan el **93% de accuracy** en Iris
- Iris setosa es la más fácil de identificar (pétalos muy pequeños)
- Versicolor y Virginica son más difíciles de separar entre sí

### Para seguir aprendiendo:
- 📚 [Scikit-learn Documentation](https://scikit-learn.org/)
- 🎓 Probá con otros datasets: `load_wine()`, `load_breast_cancer()`
- 🔍 Investigá sobre **hiperparámetros** y cómo optimizarlos